# PDF -> Markdown Converter v4 (SLR Papers)

Converts SLR research-paper PDFs to JSON + extracted images using [marker-pdf](https://github.com/VikParuchuri/marker).

**v4 changes vs v3:**
- Pillow pinned to 10.4.0 *before* anything else (fixes JPEG crash on FTS52/53/54/18/07/20/24/26/27).
- `weasyprint` installed up-front (needed for HTML-source papers like FTS18).
- Single converter init (no duplicate Gemini/OpenAI cells).
- Unified `convert_paper()` function used for both first-run batch and per-paper retries.
- **PDFs cached on Colab local disk** for fast I/O. Outputs synced back to Drive after each paper.
- Empty/dead cells removed.

**Runtime:** Go to `Runtime > Change runtime type > T4 GPU`.

**Run order:** Step 1 (Pillow) -> RESTART RUNTIME -> Step 2 onwards.


## Step 1 - Fix Pillow (run once, then RESTART RUNTIME)

Colab ships a Pillow version too new for `surya-ocr` (used inside marker). This caused JPEG-save crashes on ~10 papers in v3. Pin to 10.4.0 first.

In [1]:
# Pin Pillow to 10.4.0 (compatible with surya-ocr + marker)
!pip install -q --force-reinstall Pillow==10.4.0
import PIL
print(f"Pillow version: {PIL.__version__}")
assert PIL.__version__.startswith("10."), f"Wrong Pillow! Got {PIL.__version__}"
print("OK. NOW RESTART THE RUNTIME (Runtime -> Restart runtime), then run from Step 2.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 117.4 MB/s eta 0:00:0000:01
Pillow version: 11.3.0


AssertionError: Wrong Pillow! Got 11.3.0

## Step 2 - Mount Drive, discover PDFs, cache to local disk

Drive I/O is the bottleneck — marker re-reads each PDF many times during OCR/layout/LLM passes. We mount Drive (source of truth), discover which papers still need converting, and **copy only the pending PDFs to `/content/papers_local/`** (Colab's fast local SSD).

Outputs are written locally too, then synced to Drive after each successful paper (so a crash mid-batch doesn't lose finished work).

In [1]:
# Mount Drive, discover PDFs, cache pending ones to local disk
from google.colab import drive, files
import zipfile, os, shutil

drive.mount('/content/drive')

# Drive paths (source of truth — survive runtime restarts)
drive_input_dir  = "/content/drive/MyDrive/SLR5/papers_pdf/"
drive_output_dir = "/content/drive/MyDrive/SLR5/papers_md/"
os.makedirs(drive_input_dir,  exist_ok=True)
os.makedirs(drive_output_dir, exist_ok=True)

# Local paths (fast SSD — wiped on runtime restart)
local_input_dir  = "/content/papers_local/"
local_output_dir = "/content/papers_out/"
os.makedirs(local_input_dir,  exist_ok=True)
os.makedirs(local_output_dir, exist_ok=True)

# Upload zip if no PDFs in Drive
existing_pdfs = [f for f in os.listdir(drive_input_dir) if f.endswith('.pdf')]
if existing_pdfs:
    print(f"Found {len(existing_pdfs)} PDFs in Drive, skipping upload.")
else:
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall(drive_input_dir)
    # Flatten if zip contained a subfolder
    for root, _, fls in os.walk(drive_input_dir):
        for f in fls:
            if f.endswith('.pdf') and root != drive_input_dir:
                os.rename(os.path.join(root, f), os.path.join(drive_input_dir, f))

pdfs = sorted(f for f in os.listdir(drive_input_dir) if f.endswith('.pdf'))

# A paper is "done" if its Drive output folder contains a .json file
done = set()
for d in os.listdir(drive_output_dir):
    sub = os.path.join(drive_output_dir, d)
    if os.path.isdir(sub) and any(f.endswith('.json') for f in os.listdir(sub)):
        done.add(d)

to_convert = [f for f in pdfs if f.replace('.pdf','') not in done]
skipped    = [f for f in pdfs if f.replace('.pdf','') in done]

print(f"Total PDFs:     {len(pdfs)}")
print(f"Already done:   {len(skipped)}")
print(f"To convert:     {len(to_convert)} -> {to_convert}")

# Copy pending PDFs to local disk (fast). Skip ones already cached.
import time
copy_start = time.time()
copied, skipped_copy = 0, 0
for f in to_convert:
    src_pdf = os.path.join(drive_input_dir, f)
    dst_pdf = os.path.join(local_input_dir, f)
    if os.path.exists(dst_pdf) and os.path.getsize(dst_pdf) == os.path.getsize(src_pdf):
        skipped_copy += 1
        continue
    shutil.copy2(src_pdf, dst_pdf)
    copied += 1
elapsed = time.time() - copy_start
print(f"Cached {copied} PDFs to {local_input_dir} ({skipped_copy} already cached) in {elapsed:.1f}s")


Mounted at /content/drive
Found 18 PDFs in Drive, skipping upload.
Total PDFs:     18
Already done:   14
To convert:     4 -> ['RP44_Mazeika_2024.pdf', 'RP46_Sun_2024.pdf', 'RP47_Erdil_2025.pdf', 'RP50_Kharinaev_2025.pdf']
Cached 4 PDFs to /content/papers_local/ (0 already cached) in 4.2s


## Step 3 - Install marker-pdf + weasyprint

`weasyprint` is needed for HTML-source papers (e.g. FTS18). Both installs are skipped if already present.

In [2]:
# Install marker-pdf and weasyprint (both skip if present)
try:
    import marker
    print("marker-pdf already installed.")
except ImportError:
    print("Installing marker-pdf...")
    !pip install -q marker-pdf
    print("Done.")

try:
    import weasyprint
    print("weasyprint already installed.")
except ImportError:
    print("Installing weasyprint...")
    !pip install -q weasyprint
    print("Done.")


Installing marker-pdf...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━

## Step 4 - Initialise the converter

Default: **Gemini 2.5 Flash** (cheap, fast). To switch to OpenAI GPT-4.1-mini, uncomment the alternative block. Only one converter should be active.

In [ ]:
# Init marker converter — Gemini 2.5 Flash (default)
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser

config = {
    "output_format": "json",
    "use_llm": True,
    "gemini_api_key": "YOUR_GEMINI_API_KEY_HERE",
    "gemini_model_name": "gemini-2.5-flash",
    "timeout": 600,
    "max_retries": 10,
    "retry_wait_time": 15,
}

# --- Alternative: OpenAI GPT-4.1-mini (uncomment to use instead) ---
# config = {
#     "output_format": "json",
#     "force_ocr": True,
#     "use_llm": True,
#     "llm_service": "marker.services.openai.OpenAIService",
#     "openai_api_key": "YOUR_OPENAI_KEY",
#     "openai_model": "gpt-4.1-mini",
#     "timeout": 600,
#     "max_retries": 10,
#     "retry_wait_time": 15,
# }

config_parser = ConfigParser(config)
converter = PdfConverter(
    config=config_parser.generate_config_dict(),
    artifact_dict=create_model_dict(),
    processor_list=config_parser.get_processors(),
    renderer=config_parser.get_renderer(),
    llm_service=config_parser.get_llm_service(),
)
print(f"Converter ready ({config.get('gemini_model_name', config.get('openai_model', '?'))}).")


## Step 5 - Define `convert_paper()`

Reads the PDF from local disk (fast), writes outputs to local disk, then **syncs the output folder to Drive** so the result survives a runtime crash. Setting `force=True` wipes both local and Drive output folders for a clean re-conversion.

If the PDF isn't already cached locally, it's copied from Drive on demand.

In [ ]:
# Unified single-paper conversion — local-disk reads/writes + Drive sync after success
import time, json, os, shutil, traceback
from PIL import Image

def convert_paper(paper_name, force=False, use_converter=None):
    """Convert one paper to JSON + metadata + images.

    Reads PDF from local_input_dir (caches from Drive on demand).
    Writes outputs to local_output_dir, then syncs to drive_output_dir.

    Args:
        paper_name: e.g. "FTS04" (without .pdf)
        force: if True, wipes both local and Drive output folders first.
        use_converter: override the global `converter` (rarely needed).

    Returns: True on success, False on failure.
    """
    conv = use_converter or converter
    f = paper_name + ".pdf"

    # Lazy-cache PDF from Drive if missing locally
    drive_pdf = os.path.join(drive_input_dir, f)
    local_pdf = os.path.join(local_input_dir, f)
    if not os.path.exists(local_pdf):
        if not os.path.exists(drive_pdf):
            print(f"  [X] {drive_pdf} not found")
            return False
        shutil.copy2(drive_pdf, local_pdf)
        print(f"  [cache] copied {f} from Drive")

    local_out  = os.path.join(local_output_dir, paper_name)
    drive_out  = os.path.join(drive_output_dir, paper_name)
    json_path  = os.path.join(local_out,  paper_name + ".json")
    drive_json = os.path.join(drive_out,  paper_name + ".json")

    # Skip if already done in Drive (unless force)
    if not force and os.path.exists(drive_json):
        print(f"  [skip] {paper_name} already in Drive")
        return True

    # Clean re-conversion — wipe both local and Drive
    if force:
        for d in (local_out, drive_out):
            if os.path.exists(d):
                shutil.rmtree(d)
                print(f"  [clean] removed {d}")

    os.makedirs(local_out, exist_ok=True)
    start = time.time()
    step = "starting"

    try:
        step = "converting PDF"
        print(f"  Converting {f}...", end=" ", flush=True)
        rendered = conv(local_pdf)
        print("done.", flush=True)

        # Serialize to JSON (with two fallbacks)
        step = "serializing"
        try:
            data = json.loads(rendered.model_dump_json())
        except Exception:
            try:
                data = json.loads(rendered.json())
            except Exception:
                data = {"raw": str(rendered)}
        with open(json_path, "w", encoding="utf-8") as fp:
            json.dump(data, fp, indent=2, default=str)
        print(f"  -> JSON: {os.path.getsize(json_path):,} bytes")

        # Save metadata (table-of-contents + page stats), with fallbacks
        step = "saving metadata"
        if hasattr(rendered, "metadata"):
            meta = rendered.metadata
            meta_data = None
            for attempt in (
                lambda: json.loads(meta.model_dump_json()),
                lambda: json.loads(meta.json()),
            ):
                try:
                    meta_data = attempt()
                    break
                except Exception:
                    continue
            if meta_data is None:
                meta_data = {}
                if hasattr(meta, "table_of_contents") and meta.table_of_contents:
                    meta_data["table_of_contents"] = [
                        {
                            "title": getattr(t, "title", ""),
                            "heading_level": getattr(t, "heading_level", None),
                            "page_id": getattr(t, "page_id", None),
                        } for t in meta.table_of_contents
                    ]
                if hasattr(meta, "page_stats") and meta.page_stats:
                    meta_data["page_stats"] = [
                        {
                            "page_id": getattr(p, "page_id", None),
                            "text_extraction_method": getattr(p, "text_extraction_method", None),
                            "block_counts": getattr(p, "block_counts", None),
                        } for p in meta.page_stats
                    ]
                if not meta_data:
                    meta_data = {"raw": str(meta)}
            meta_path = os.path.join(local_out, paper_name + "_metadata.json")
            with open(meta_path, "w", encoding="utf-8") as fp:
                json.dump(meta_data, fp, indent=2, default=str)
            print(f"  -> metadata saved")

        # Extract and save images
        step = "extracting images"
        images = getattr(rendered, "images", {}) or {}
        for img_name, img_data in images.items():
            step = f"saving image {img_name}"
            img_path = os.path.join(local_out, img_name)
            if isinstance(img_data, Image.Image):
                img_data.save(img_path)
            else:
                with open(img_path, "wb") as fp:
                    fp.write(img_data)
        print(f"  -> {len(images)} images saved (local)")

        # Sync to Drive (atomic-ish: write to staging then rename)
        step = "syncing to Drive"
        sync_start = time.time()
        if os.path.exists(drive_out):
            shutil.rmtree(drive_out)
        shutil.copytree(local_out, drive_out)
        print(f"  -> synced to Drive in {time.time() - sync_start:.1f}s")

        elapsed = time.time() - start
        print(f"  [OK] {paper_name} ({elapsed:.1f}s total)")
        return True

    except Exception as e:
        elapsed = time.time() - start
        print(f"\n  [FAIL] {paper_name} at step: {step} ({elapsed:.1f}s)")
        print(f"    Error: {e}")
        print(f"    Traceback:\n{''.join(traceback.format_exc())}")
        return False

print("convert_paper() defined.")


convert_paper() defined.


## Step 6 - Batch convert all pending papers

Iterates over `to_convert` and calls `convert_paper()` for each. Already-done papers (per Drive) are skipped automatically. Failures are collected so you can retry them in Step 7.

In [ ]:
# Batch convert all pending PDFs (fast: reads from local disk)
import time

failed = []
batch_start = time.time()

for i, f in enumerate(to_convert, 1):
    paper = f.replace(".pdf", "")
    print(f"\n[{i}/{len(to_convert)}] {paper}")
    if not convert_paper(paper, force=False):
        failed.append(paper)

elapsed = time.time() - batch_start
print(f"\n{'='*60}")
print(f"Batch done in {elapsed:.1f}s. {len(to_convert) - len(failed)}/{len(to_convert)} OK.")
if failed:
    print(f"Failed: {failed}")
    print(f"Retry them in Step 7 with: convert_paper('FTSxx', force=True)")



[1/18] RP08_Aminabadi_2022
  Converting RP08_Aminabadi_2022.pdf... 

LLMTableProcessor running: 100%|██████████| 3/3 [00:17<00:00,  5.96s/it]
LLM processors running: 0it [00:00, ?it/s]
Running LLMSectionHeaderProcessor: 100%|██████████| 1/1 [04:12<00:00, 252.54s/it]


done.
  -> JSON: 1,223,642 bytes
  -> metadata saved
  -> 0 images saved (local)
  -> synced to Drive in 0.1s
  [OK] RP08_Aminabadi_2022 (416.3s total)

[2/18] RP11_Dettmers_2022
  Converting RP11_Dettmers_2022.pdf... 

LLMTableProcessor running: 100%|██████████| 10/10 [04:12<00:00, 25.25s/it]
LLM processors running: 0it [00:00, ?it/s]
Running LLMSectionHeaderProcessor: 100%|██████████| 1/1 [00:16<00:00, 16.09s/it]


done.
  -> JSON: 1,291,755 bytes
  -> metadata saved
  -> 0 images saved (local)
  -> synced to Drive in 0.0s
  [OK] RP11_Dettmers_2022 (423.1s total)

[3/18] RP14_Yao_2022
  Converting RP14_Yao_2022.pdf... 

...

done.
  -> JSON: 1,606,897 bytes
  -> metadata saved
  -> 0 images saved (local)
  -> synced to Drive in 0.0s
  [OK] RP40_Dettmers_2023 (271.2s total)

[12/18] RP41_Xu_2024
  Converting RP41_Xu_2024.pdf... 

LLMTableProcessor running: 100%|██████████| 33/33 [05:05<00:00,  9.24s/it]
LLM processors running: 0it [00:00, ?it/s]
Running LLMSectionHeaderProcessor: 100%|██████████| 1/1 [00:27<00:00, 27.96s/it]


done.
  -> JSON: 10,320,193 bytes
  -> metadata saved
  -> 0 images saved (local)
  -> synced to Drive in 0.1s
  [OK] RP41_Xu_2024 (800.7s total)

[13/18] RP42_Lin_2024
  Converting RP42_Lin_2024.pdf... 

Recognizing Text:   6%|▋         | 16/251 [01:23<06:47,  1.73s/it] 

: 

## Step 7 - Retry individual papers

Use this for papers that failed in the batch. `force=True` wipes the partial output (local + Drive) before re-running. Examples below mirror the problem papers seen in v3.

In [ ]:
# Retry one paper at a time. force=True wipes any partial output first.
# Edit the paper name and run.

#convert_paper("RP14_Lin_2023", force=True)
#convert_paper("RP50_Mi_2025", force=True)

# Other v3 problem papers — uncomment to retry:
# for p in ["FTS07", "FTS13", "FTS18", "FTS19", "FTS20", "FTS24",
#           "FTS26", "FTS27", "FTS32", "FTS36", "FTS38",
#           "FTS52", "FTS53", "FTS54"]:
#     print(f"\n--- {p} ---")
#     convert_paper(p, force=True)


## Step 8 - Final verification (auto-discover targets)

Reads the list of target papers from `drive_input_dir` (the PDFs you actually loaded), then checks each one for JSON output, metadata, and images in `drive_output_dir`. Works regardless of the naming scheme (FTS##, RP##_Author_Year, etc.).

Lists any missing or empty outputs at the end so you can retry them in Step 7.

In [ ]:
# Verify each input PDF has a valid output in Drive (auto-discovered targets)
import os

# Targets = whatever PDFs are sitting in the input dir (RP##_Author_Year, FTS##, etc.)
targets = sorted(
    f.replace(".pdf", "") for f in os.listdir(drive_input_dir) if f.endswith(".pdf")
)
name_w = max((len(t) for t in targets), default=10)

print(f"Verifying {len(targets)} papers from {drive_input_dir}\n")
print(f"{'Paper':<{name_w}}  {'JSON':>4}  {'Size':>14}  {'Meta':>4}  {'Images':>6}")
print("-" * (name_w + 38))

missing, empty = [], []
total_bytes, total_images = 0, 0

for p in targets:
    pdir = os.path.join(drive_output_dir, p)
    json_path = os.path.join(pdir, p + ".json")
    meta_path = os.path.join(pdir, p + "_metadata.json")

    has_json = os.path.exists(json_path)
    has_meta = os.path.exists(meta_path)
    json_sz  = os.path.getsize(json_path) if has_json else 0
    img_count = (
        len([f for f in os.listdir(pdir) if f.endswith(('.png','.jpg','.jpeg','.webp'))])
        if os.path.exists(pdir) else 0
    )

    if not has_json:
        missing.append(p)
    elif json_sz < 1024:    # suspiciously small => likely a failure
        empty.append(p)

    total_bytes  += json_sz
    total_images += img_count

    status = "OK" if has_json else "X"
    meta_s = "OK" if has_meta else "X"
    print(f"{p:<{name_w}}  {status:>4}  {json_sz:>12,} B  {meta_s:>4}  {img_count:>6}")

print(f"\n{'='*(name_w + 38)}")
print(f"Total: {len(targets)} papers, {total_bytes:,} bytes JSON, {total_images} images")

if not missing and not empty:
    print(f"All {len(targets)} papers have valid JSON output in Drive.")
else:
    if missing:
        print(f"Missing ({len(missing)}): {missing}")
    if empty:
        print(f"Suspiciously small (<1 KB) ({len(empty)}): {empty}")
    bad = missing + empty
    print(f"\nRetry in Step 7:")
    for p in bad:
        print(f"  convert_paper({p!r}, force=True)")


## Step 9 - Zip and download

Bundles `drive_output_dir` into a zip and downloads it to your local machine.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/papers_md", "zip", drive_output_dir)
files.download("/content/papers_md.zip")
print("Downloading papers_md.zip")
